# 🚗🚚🚙🚍**CurvantML: Previsão de Condução de Risco em Curvas**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
plt.rcParams.update({'font.size': 12}) # Ajusta o tamanho da fonte das imagens para 12

## 📂Carregando os dados tratados

In [ ]:
df = pd.read_parquet('data/eletro_rjdf_serra.parquet')
df.head()

,lat,lon,vehicle_speed,accel_x,accel_y,accel_z,engine_rpm,mass_air_flow,absolute_throttle_pos,fuel_level,...,fuel_remaining,instant_fuel_economy,calculated_load_value,fuel_rail_pressure,accelerator_pos_d,accelerator_pos_e,time,accelerator_pedal_pos_d,accelerator_pedal_pos_e,fuel_rail_pressure_vacuum
0,-22.93086,-43.97891,68.0,0.363016,0.382445,-0.194902,2012.25,26.33,82.35294,70.19608,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-22.93093,-43.97915,70.0,-0.433087,0.174808,0.165078,2078.75,27.53,83.92157,70.19608,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-22.93096,-43.97928,71.0,-0.063610,0.018245,0.090633,2143.75,28.70,38.03922,72.54902,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-22.93102,-43.97949,72.0,0.241251,0.234123,1.493305,2104.75,18.20,34.90196,72.54902,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-22.93108,-43.97971,72.0,0.440336,-0.698661,-1.040103,1605.50,18.39,36.47059,72.54902,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---



## **📈Detectar curvas**

A **Curvatura** $\kappa(t)$ de uma curva $C:{I} → \mathbb{R}^2$, $I\subset \mathbb{R}$ é dada por

$$\kappa(t) = \frac{|x'(t) y''(t) - y'(t) x''(t)|}{\left( (x'(t))^2 + (y'(t))^2 \right)^{3/2}}
$$

O **Filtro Gaussiano** realiza a convolução de um sinal com a função gaussiana, resultando em um novo sinal suavizado. A função gaussiana é definida como:

$$
G(x) = \frac{1}{\sqrt{2\pi \sigma^2}} e^{-\frac{x^2}{2\sigma^2}}
$$

onde:
- $ \sigma $ é o desvio padrão, que determina a largura da distribuição.

Para aplicar o filtro gaussiano a um sinal 1D $ f(t) $, a convolução é realizada da seguinte forma:

$$
g(t) = \int_{-\infty}^{\infty} f(\tau) G(t - \tau) d\tau
$$

onde:
- $ g(t) $ é o sinal suavizado.
- $ f(t) $ é o sinal original.
- $ G(t - \tau) $ é a função gaussiana centrada em $t$.

#### Detecção

In [ ]:
from src.curve_detection import detectar_curvas, identificar_trechos_curvos
from src.driving_analysis import detectar_conducao_perigosa
from src.features import extrair_features
from src.models import aplicar_modelos_ml
from src.visualization import plotar_trajeto_conducao
from src.utils import contar_curvas

In [ ]:
df_sample = [df.query(f'id_route == "{trip}"') for trip in list(df.id_route.unique())[0:5]]

In [ ]:
data_curves = []
for dt in df_sample:
    data_curve_traj = detectar_curvas(dt, sigma=2)
    data_curves.append(data_curve_traj)
dfs_curves = pd.concat(data_curves)

In [ ]:
contar_curvas(dfs_curves)

{'obd-15-spin-trajeto-t1': 105,
 'obd-16-van-circuito-t1': 26,
 'obd-16-van-circuito-t2': 54,
 'obd-16-van-trajeto-t3': 5,
 'obd-17-spin-trajeto-t1': 2}

---

## 🚨**Condução de risco no contexto veicular**

**DEFINIÇÕES DE COMPORTAMENTOS DE CONDUÇÃO PERIGOSOS**

(_Li et al. (2016) Dangerous driving behavior detection using smartphone sensors_)

O objetivo é reconhecer comportamentos de condução perigosos. Três tipos de comportamento de risco típico são selecionados e definidos:

1. Acelerar ou desacelerar anormalmente: aumentar ou diminuir
a velocidade do veículo bruscamente em um curto período, como a
variância da velocidade ser maior que 30 km/h em 10 segundos.

2. Direção: virar à esquerda ou à direita mais de 0,7 rad a uma
velocidade acima de 30 km/h, o que pode causar capotamento.
Vamos

3. Zigue-zague: dirigir em forma de S, o veículo
muda de faixa com muita frequência, o que pode causar engarrafamentos
ou arranhões no veículo.

`Aceleração centripeta` e `Aceleração Absoluta`

In [ ]:
dfs_curves['ctp_accel'] = (dfs_curves['vehicle_speed']/3.6)**2/dfs_curves['raio_curvatura'] # m/s**2
dfs_curves['abs_accel'] = np.sqrt(dfs_curves['accel_x']**2 + dfs_curves['accel_y']**2)

In [ ]:
df_analysis = []

for traj in dfs_curves.id_route.unique():
  df = dfs_curves.query(f'id_route == "{traj}"')
  df_analysis += [detectar_conducao_perigosa(df, janela_tempo = 10)]

df_analysis = pd.concat(df_analysis)

In [ ]:
tjs = df_analysis.id_route.unique()
for traj in tjs:

  # trajetos ruins:
  # obd-17-van-circuito-t1
  # obd-17-van-circuito-t2

  dteste = df_analysis.query(f'id_route == "{traj}"')
  plotar_trajeto_conducao(dteste)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
total_time = 0
for traj in df_analysis.id_route.unique():
  df = df_analysis.query(f'id_route == "{traj}"')
  print(df.time_sec.max()/60/60,traj)
  total_time += df.time_sec.max()

print(f'Tempo Total: {total_time/(60*60)}')

1.7252947222222221 obd-15-spin-trajeto-t1
0.863695 obd-16-van-circuito-t1
1.2976822222222222 obd-16-van-circuito-t2
0.20293361111111113 obd-16-van-trajeto-t3
0.12088638888888889 obd-17-spin-trajeto-t1
0.10279861111111112 obd-17-spin-trajeto-t2
0.3887216666666667 obd-18-van-trajeto-t2
2.388967777777778 obd-18-van-trajeto-t3
0.35023499999999996 tabela_final-16-spin-circuito-t1
0.3635627777777778 tabela_final-16-spin-trajeto-t2
23.888056944444443 tabela_final-16-van-trajeto-t1
1.7170363888888889 tabela_final-16-van-trajeto-t2
1.4415483333333332 tabela_final-18-van-trajeto-t1
3.432377777777778 tabela_final-19-spin-trajeto-t1
1.1211991666666667 CSVLog_20240920_044510
0.6926452777777778 CSVLog_20240920_062459
1.5506633333333333 CSVLog_20240920_073017
0.2375 CSVLog_20240920_090744
1.36076 CSVLog_20240920_093107
1.891075 CSVLog_20240920_115708
1.4972375000000002 CSVLog_20240920_135802
0.3247186111111111 CSVLog_20240921_070315
1.5347244444444443 CSVLog_20240921_081801
0.7850066666666666 CSVLog_

---

### ✨APLICANDO OS ALGORITIMOS DE ML

## 📥📤Detecção de condução perigosa em **curvas**

Identificar trechos curvos

📊Média de algumas variáveis por trajeto

📥*Features*

In [ ]:
df_analysis[['aceleracao_anormal', 'direcao_perigosa', 'zigue_zague']] = df_analysis[['aceleracao_anormal', 'direcao_perigosa', 'zigue_zague']].astype(int)

In [ ]:
dfs_bons = df_analysis.copy()
dfs_bons = identificar_trechos_curvos(dfs_bons)

In [ ]:
teste_com_trechos_bons = extrair_features(dfs_bons)

In [ ]:
tab_result = pd.DataFrame(
    {
        'Manobra': ['Segura', 'Perigosa'],
        'Conducão': [teste_com_trechos_bons.manobra.value_counts()[0],
                     teste_com_trechos_bons.manobra.value_counts()[1]],
        'Aceleração anormal': [teste_com_trechos_bons.manobra_accel_perigo.value_counts()[0],
                               teste_com_trechos_bons.manobra_accel_perigo.value_counts()[1]],
        'Direção perigosa': [teste_com_trechos_bons.manobra_dir_perigosa.value_counts()[0],
                             teste_com_trechos_bons.manobra_dir_perigosa.value_counts()[1]
                             ],
        'Zigue-zague': [teste_com_trechos_bons.manobra_zigue_zague.value_counts()[0],
                        teste_com_trechos_bons.manobra_zigue_zague.value_counts()[1]]
    }
)

In [ ]:
tab_result.set_index('Manobra', inplace=True)
tab_result

,Conducão,Aceleração anormal,Direção perigosa,Zigue-zague
Manobra,,,,
Segura,1146,1942,2126,2618
Perigosa,1983,1187,1003,511


Lembra que a `conducao` é `True` (perigosa) se `aceleracao anormal` é `True` **OU** `direção` é `True` **OU** `zigue-zague` é `True`.

In [ ]:
teste_com_trechos_bons.loc[:,['manobra_accel_perigo', 'manobra_dir_perigosa', 'manobra_zigue_zague']].value_counts().to

manobra_accel_perigo  manobra_dir_perigosa  manobra_zigue_zague
0                     0                     0                      606
                      1                     0                       90
                      0                     1                       87
1                     0                     0                       24
                                            1                       21
                      1                     0                        8
0                     1                     1                        7
1                     1                     1                        3
Name: count, dtype: int64

In [ ]:
tab_result.to_latex('tab_result.tex', index=True)

📤Modelos

In [ ]:
df_ml = teste_com_trechos_bons.copy()

In [ ]:
aplicar_modelos_ml(df_ml, plot_cm=True)

---

# ▶ Redes Neurais ◀

Melhor MLP

---

## Testando outros modelos de Redes Neurais